### REad BTC Data

In [1]:
import pandas as pd

df = pd.read_csv("BTCUSDT.csv", sep='|')
df = df.rename(columns={'1502942400': 'timestamp', '4261.48': 'open', '4261.48.1': 'high', '4261.48.2': 'low', '4261.48.3': 'close', '1.775183': 'btc_volume_sell', '320.390851': 'usd_volume_buy', '0.075183': 'btc_volume_buy', '7564.906851': 'usd_volume_sell', '3': 'activity'}) 

df

,timestamp,open,high,low,close,btc_volume_sell,usd_volume_buy,btc_volume_buy,usd_volume_sell,activity
0,1502942460,4261.48,4261.48,4261.48,4261.48,0.000000,0.000000,0.000000,0.000000e+00,0
1,1502942520,4280.56,4280.56,4280.56,4280.56,0.261074,1117.542921,0.261074,1.117543e+03,2
2,1502942580,4261.48,4261.48,4261.48,4261.48,0.012008,51.171852,0.012008,5.117185e+01,3
3,1502942640,4261.48,4261.48,4261.48,4261.48,0.140796,599.999338,0.140796,5.999993e+02,1
4,1502942700,4261.48,4261.48,4261.48,4261.48,0.000000,0.000000,0.000000,0.000000e+00,0
...,...,...,...,...,...,...,...,...,...,...
3997969,1743334260,83149.78,83184.65,83132.43,83184.65,3.664100,236173.309690,2.840050,3.047000e+05,1811
3997970,1743334320,83184.65,83184.66,83141.85,83141.85,2.287810,60844.870800,0.731660,1.902545e+05,1300
3997971,1743334380,83141.85,83141.86,83100.00,83104.94,3.499020,164623.561700,1.980910,2.908151e+05,1411
3997972,1743334440,83106.00,83110.76,83038.02,83055.34,16.144520,231423.308990,2.786300,1.341206e+06,4368


### Derive volatility

In [2]:
import numpy as np

# volatility
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df.set_index('datetime', inplace=True)
df['log_return'] = np.log(df['close'] / df['close'].shift(1))
window_size = 30 * 60 * 60
df['30d_volatility'] = df['log_return'].rolling(window=window_size).std() * np.sqrt(window_size)
df[['close', 'log_return', '30d_volatility']].tail(10)

# avg volume, avg buy volume, avg sell volume
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df.set_index('datetime', inplace=True)
df['usd_total_volume'] = df['usd_volume_buy'] + df['usd_volume_sell']
df['7d_usd_volume_buy_avg'] = df['usd_volume_buy'].rolling(window=7*24*60, center=False).mean()
df['7d_usd_volume_sell_avg'] = df['usd_volume_sell'].rolling(window=7*24*60, center=False).mean()
df['7d_usd_volume_avg'] = df['usd_total_volume'].rolling(window=7*24*60, center=False).mean()

max_vol = max(df[df['30d_volatility'].notna()]['30d_volatility'])
print(max_vol)

window = 4 * 60
#first value most important, last
df['forward_usd_volume_buy_avg'] =  df['usd_volume_buy'].ewm(span=window, adjust=False).mean()
df['forward_usd_volume_sell_avg'] =  df['usd_volume_sell'].ewm(span=window, adjust=False).mean()
df['forward_usd_volume_avg'] =  df['usd_total_volume'].ewm(span=window, adjust=False).mean()

df['price_change_4h'] = (df['close'].shift(-15) - df['close'].shift(window) ) / df['close']
df['price_change_4h_tanh'] = np.tanh(df['price_change_4h'])
df['price_change_4h_volatility_tanh'] = np.tanh(df['price_change_4h'] * (max_vol - df['30d_volatility']))

df['price_change_4h_ema'] = df['price_change_4h_tanh'].ewm(span=60, adjust=False).mean()
df['price_change_4h_volatility_ema'] = df['price_change_4h_volatility_tanh'].ewm(span=60, adjust=False).mean()

df[['close','usd_volume_buy', 'forward_usd_volume_buy_avg', '7d_usd_volume_buy_avg', 'price_change_4h']]

0.9900359831658481


,close,usd_volume_buy,forward_usd_volume_buy_avg,7d_usd_volume_buy_avg,price_change_4h
datetime,,,,,
2017-08-17 04:01:00,4261.48,0.000000,0.000000,NaN,NaN
2017-08-17 04:02:00,4280.56,1117.542921,9.274215,NaN,NaN
2017-08-17 04:03:00,4261.48,51.171852,9.621913,NaN,NaN
2017-08-17 04:04:00,4261.48,599.999338,14.521311,NaN,NaN
2017-08-17 04:05:00,4261.48,0.000000,14.400802,NaN,NaN
...,...,...,...,...,...
2025-03-30 11:31:00,83184.65,236173.309690,216449.362461,534779.603856,NaN
2025-03-30 11:32:00,83141.85,60844.870800,215158.038878,534767.281993,NaN
2025-03-30 11:33:00,83104.94,164623.561700,214738.665624,534748.305245,NaN


In [3]:
from math import sqrt
import numpy as np
import pandas as pd

df['raw_volume_surge'] = (
    df['forward_usd_volume_buy_avg'] / df['7d_usd_volume_buy_avg'] -
    df['forward_usd_volume_sell_avg'] / df['7d_usd_volume_sell_avg']
)

# adjust by volatility
max_vol = max(df[df['30d_volatility'].notna()]['30d_volatility'])
print(max_vol)

df['adjusted_surge'] = np.sqrt(df['raw_volume_surge'] * (max_vol - df['30d_volatility'] + 1e-8) )
df['adjusted_surge_2'] = np.sqrt((df['raw_volume_surge'] * (max_vol - df['30d_volatility'] + 1e-8)) * abs(df['price_change_4h']))

df['adjusted_surge'] = (df['raw_volume_surge'] * (max_vol - df['30d_volatility'] + 1e-8) )
df['adjusted_surge_2'] = ((df['raw_volume_surge'] * (max_vol - df['30d_volatility'] + 1e-8)) * abs(df['price_change_4h']))

df['volume_surge'] = np.tanh(df['raw_volume_surge'])
df['volume_surge_volatility'] = np.tanh(df['adjusted_surge'])
df['volume_surge_price'] = np.tanh(df['adjusted_surge_2'])


dff = df.copy()
dff = dff[dff['volume_surge'].notna()]
dff = dff[dff['volume_surge_volatility'].notna()]
print(len(dff))
dff[['timestamp', 'volume_surge', 'volume_surge_volatility', 'volume_surge_price', 'price_change_4h']].sort_values(by='volume_surge_price',ascending= True).head(20)


0.9900359831658481


/tmp/claude-1000/-home-kosmo-Documents-Magis-submit-src-vibe--Copy-/a3bca740-2cff-4124-8944-29676ab58f7a/scratchpad/venv310/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/claude-1000/-home-kosmo-Documents-Magis-submit-src-vibe--Copy-/a3bca740-2cff-4124-8944-29676ab58f7a/scratchpad/venv310/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


3889974


,timestamp,volume_surge,volume_surge_volatility,volume_surge_price,price_change_4h
datetime,,,,,
2019-09-24 19:29:00,1569353340,-0.841425,-0.671569,-0.159339,-0.197529
2019-09-24 19:32:00,1569353520,-0.845473,-0.676619,-0.144121,-0.176376
2019-09-24 19:45:00,1569354300,-0.887913,-0.731368,-0.143082,-0.154638
2019-09-24 19:44:00,1569354240,-0.898435,-0.748168,-0.141256,-0.146790
2019-09-24 19:33:00,1569353580,-0.843921,-0.674666,-0.140776,-0.172982
2019-09-24 19:28:00,1569353280,-0.831580,-0.659486,-0.139402,-0.177188
2020-03-12 10:46:00,1584009960,-0.592384,-0.454312,-0.137526,-0.282385
2019-09-24 19:30:00,1569353400,-0.849472,-0.681724,-0.136356,-0.164852
2019-09-24 19:31:00,1569353460,-0.844828,-0.675821,-0.134303,-0.164503


In [4]:
# window_4h = int(4 * 60 / 2)  # 2 hours before and after

# max_vol = max(df[df['30d_volatility'].notna()]['30d_volatility'])
# print(max_vol)

# # momentum change (as percentage)
# df['momentum_change'] = (df['close'].shift(-window_4h) - df['close'].shift(window_4h)) / df['close'].shift(window_4h)
# df['momentum_change_volatility'] = ((df['close'].shift(-window_4h) - df['close'].shift(window_4h)) / df['close'].shift(window_4h)) * (max_vol - df['30d_volatility'] + 1e-8) 

# # smooth this using an EMA to reduce noise
# df['momentum_ema'] = df['momentum_change'].ewm(span=60, adjust=False).mean()
# df['momentum_volatility_ema'] = df['momentum_change_volatility'].ewm(span=60, adjust=False).mean()

# # Inspect the results
# df[['close', 'momentum_change', 'momentum_change_volatility', 'momentum_volatility_ema', 'momentum_ema']].sort_values(by='momentum_volatility_ema',ascending= False).head(20)

### connect with headlines

In [5]:
df_news = pd.read_csv("ok.csv")
df_news.rename(columns={'timestamp': 'newsTimestamp'}, inplace=True)
df_news.rename(columns={'from': 'source'}, inplace=True)
df_news.rename(columns={'text': 'title'}, inplace=True)
df_news.rename(columns={'date_unixtime': 'timestamp'}, inplace=True)
df_news.rename(columns={'date': 'datetime'}, inplace=True)
df_news = df_news.drop('Unnamed: 0',axis='columns')
df_news['newsTimestamp'] = pd.to_datetime(df_news['datetime']).astype(int) // 10**9
df['timestamp'] = df['timestamp'].astype(int)
df = df.sort_values(by='timestamp')

df_news = df_news[df_news['title'].str.len() > 3]

news_times = df_news['newsTimestamp'].values
df_timestamps = df['timestamp'].values

# Find the closest previous timestamp index (open price before news)
news_indices = np.searchsorted(df_timestamps, news_times, side='right') - 1
news_indices = np.clip(news_indices, 0, len(df) - 1)  # Ensure valid indices

df_news['open_before_news'] = df.iloc[news_indices]['open'].values
df_news['30d_volatility'] = df.iloc[news_indices]['30d_volatility'].values
df_news['volume_surge'] = df.iloc[news_indices]['volume_surge'].values
df_news['volume_surge_volatility'] = df.iloc[news_indices]['volume_surge_volatility'].values
df_news['volume_surge_price'] = df.iloc[news_indices]['volume_surge_price'].values
# df_news['momentum_change'] = df.iloc[news_indices]['momentum_change'].values
# df_news['momentum_change_volatility'] = df.iloc[news_indices]['momentum_change_volatility'].values
# df_news['momentum_ema'] = df.iloc[news_indices]['momentum_ema'].values
# df_news['momentum_volatility_ema'] = df.iloc[news_indices]['momentum_volatility_ema'].values
df_news['price_change_4h'] = df.iloc[news_indices]['price_change_4h'].values
df_news['price_change_4h_tanh'] = df.iloc[news_indices]['price_change_4h_tanh'].values
df_news['price_change_4h_volatility_tanh'] = df.iloc[news_indices]['price_change_4h_volatility_tanh'].values
df_news['price_change_4h_ema'] = df.iloc[news_indices]['price_change_4h_ema'].values
df_news['price_change_4h_volatility_ema'] = df.iloc[news_indices]['price_change_4h_volatility_ema'].values



print(len(df_news))
print(df_news.columns)

df_news.sort_values(by='price_change_4h',ascending= True).head(10)

20278
Index(['datetime', 'timestamp', 'title', 'source', 'newsTimestamp',
       'open_before_news', '30d_volatility', 'volume_surge',
       'volume_surge_volatility', 'volume_surge_price', 'price_change_4h',
       'price_change_4h_tanh', 'price_change_4h_volatility_tanh',
       'price_change_4h_ema', 'price_change_4h_volatility_ema'],
      dtype='object')


,datetime,timestamp,title,source,newsTimestamp,open_before_news,30d_volatility,volume_surge,volume_surge_volatility,volume_surge_price,price_change_4h,price_change_4h_tanh,price_change_4h_volatility_tanh,price_change_4h_ema,price_change_4h_volatility_ema
10389,2020-03-12T13:28:15,1584012495,Cost of liquidity spiking,unfolded.,1584019695,5990.00,0.326517,-0.258797,-0.173926,-0.045782,-0.260731,-0.254979,-0.171294,-0.228703,-0.153522
10383,2020-03-12T12:30:00,1584009000,Tokyo police arrest two for taking possession ...,unfolded.,1584016200,6085.76,0.325006,-0.388172,-0.265881,-0.064314,-0.236406,-0.232098,-0.155934,-0.205916,-0.138898
10384,2020-03-12T12:35:27,1584009327,Another leg up for implied volatility,unfolded.,1584016527,6119.76,0.325041,-0.379887,-0.259853,-0.062583,-0.235626,-0.231360,-0.155420,-0.210323,-0.141779
10385,2020-03-12T12:54:14,1584010454,new ฿ daily low: \n\nBitstamp - $5965\nBitfine...,unfolded.,1584017654,5967.72,0.325572,-0.375274,-0.256317,-0.061072,-0.233243,-0.229104,-0.153753,-0.216731,-0.145777
10382,2020-03-12T11:44:49,1584006289,LocalBitcoins seller charged with money launde...,unfolded.,1584013489,5926.26,0.318579,-0.599217,-0.433824,-0.107187,-0.231598,-0.227545,-0.154267,-0.200906,-0.137782
10381,2020-03-12T11:44:49,1584006289,CME Group to close Chicago trading floor as a ...,unfolded.,1584013489,5926.26,0.318579,-0.599217,-0.433824,-0.107187,-0.231598,-0.227545,-0.154267,-0.200906,-0.137782
10388,2020-03-12T13:13:06,1584011586,3rd biggest daily liquidations the last 2 year...,unfolded.,1584018786,6027.92,0.326381,-0.284981,-0.192098,-0.044395,-0.228383,-0.224494,-0.150418,-0.219937,-0.147670
10386,2020-03-12T12:58:20,1584010700,Whopping $520m liquidated so far on BitMEX and...,unfolded.,1584017900,6068.84,0.326137,-0.343383,-0.233251,-0.053154,-0.223902,-0.220234,-0.147563,-0.217591,-0.146292
10387,2020-03-12T13:03:58,1584011038,BitMEX OI $670m,unfolded.,1584018238,6061.27,0.326275,-0.315866,-0.213735,-0.047821,-0.220457,-0.216954,-0.145295,-0.217733,-0.146305
10380,2020-03-12T11:19:52,1584004792,(ERC20) balances on exchanges by +125% within ...,unfolded.,1584011992,6354.75,0.315167,-0.584309,-0.423110,-0.086952,-0.193083,-0.190719,-0.129574,-0.164060,-0.114164


In [6]:
# import re
# import numpy as np
# from sklearn.metrics import classification_report, r2_score
# from sklearn.metrics import mean_squared_error
# from sklearn.model_selection import train_test_split
# from sklearn.feature_extraction.text import TfidfVectorizer
# from nltk.corpus import stopwords
# from nltk.tokenize import word_tokenize
# from nltk.stem import WordNetLemmatizer
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.svm import SVR

# df_news = df_news[df_news['price_change_4h'].notna()]
# df_news = df_news[df_news['volume_surge_volatility'].notna()]

# stop_words = set(stopwords.words('english'))
# lemmatizer = WordNetLemmatizer()

# def clean_text(text):
#     text = text.lower()
#     text = re.sub(r'\d+', '', text)  # Remove numbers
#     text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
#     words = word_tokenize(text)
#     words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
#     #print(' '.join(words))
#     return ' '.join(words)

# weights = ['volume_surge', 'volume_surge_volatility', 'volume_surge_price', 
#             'price_change_4h_tanh', 'price_change_4h_volatility_tanh' ,
#               'price_change_4h_ema', 'price_change_4h_volatility_ema']

# column_for_correlation = 'price_change_4h'
# cleaned = df_news['title'].apply(clean_text)
# vectorizer =  TfidfVectorizer(max_df=0.05, min_df=1,ngram_range=(1,7),sublinear_tf=False)
# x = vectorizer.fit_transform(cleaned)

# for w in weights:    
#     y = df_news[w]
#     x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=13)

#     svm_clf = SVR(kernel='linear', C=0.1, epsilon=0.5, gamma='scale')  
#     svm_clf.fit(x_train, y_train)
#     y_pred = svm_clf.predict(x_test)
#     mse = mean_squared_error(y_test, y_pred)
#     print(w)
#     print(f"Mean Squared Error (MSE): {mse}")
#     rmse = np.sqrt(mse)
#     print(f"Root Mean Squared Error (RMSE): {rmse}")
#     r2 = r2_score(y_test, y_pred)
#     print(f"R² Score: {r2}")
#     print('')

In [7]:
from sklearn.preprocessing import MinMaxScaler


selected_columns = ['timestamp', 'title', 'source', 'volume_surge_price', 'price_change_4h']
df_save = df_news[selected_columns]
df_save = df_save.drop(df_save.nlargest(10, 'volume_surge_price').index)
df_save = df_save.drop(df_save.nsmallest(10, 'volume_surge_price').index)


df_save.sort_values('volume_surge_price', ascending=True).head(30)

neg_mask = df_save['volume_surge_price'] < 0
pos_mask = df_save['volume_surge_price'] > 0

scaler_neg = MinMaxScaler(feature_range=(-1, 0))
scaler_pos = MinMaxScaler(feature_range=(0, 1))

df_save.loc[neg_mask, 'volume_surge_price'] = scaler_neg.fit_transform(df_save.loc[neg_mask, ['volume_surge_price']])
df_save.loc[pos_mask, 'volume_surge_price'] = scaler_pos.fit_transform(df_save.loc[pos_mask, ['volume_surge_price']])


#df_save.sort_values('volume_surge_price', ascending=False).head(30)
df_save.to_csv('output.csv', index=False)

In [8]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.discriminant_analysis import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.stats import pearsonr

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVR

df_news = df_news[df_news['price_change_4h'].notna()]
df_news = df_news[df_news['volume_surge_volatility'].notna()]

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

cleaned_titles = df_news['title'].apply(clean_text)
vectorizer = TfidfVectorizer(max_df=0.1, min_df=1, ngram_range=(1,7), sublinear_tf=False)
X = vectorizer.fit_transform(cleaned_titles)

weights = ['volume_surge', 'volume_surge_volatility', 'volume_surge_price',
            'price_change_4h_tanh', 'price_change_4h_volatility_tanh' , 'price_change_4h_ema', 'price_change_4h_volatility_ema']
column_for_correlation = 'price_change_4h'

results = []

# Loop through each target
for w in weights:


    df_copy = df_news.copy()

    col = w
    df_copy = df_copy.drop(df_copy.nlargest(3, col).index)
    df_copy = df_copy.drop(df_copy.nsmallest(3, col).index)

    cleaned_titles = df_copy['title'].apply(clean_text)
    vectorizer = TfidfVectorizer(max_df=0.1, min_df=1, ngram_range=(1,7), sublinear_tf=False)
    X = vectorizer.fit_transform(cleaned_titles)

    neg_mask = df_copy[col] < 0
    pos_mask = df_copy[col] > 0
    neg_scaler = MinMaxScaler(feature_range=(-1, 0))
    pos_scaler = MinMaxScaler(feature_range=(0, 1))

    scaled_s = pd.Series(index=df_copy.index, dtype='float64')
    if neg_mask.any():
        scaled_s.loc[neg_mask.index[neg_mask]] = neg_scaler.fit_transform(
            df_copy.loc[neg_mask, col].values.reshape(-1, 1)
        ).flatten()
    if pos_mask.any():
        scaled_s.loc[pos_mask.index[pos_mask]] = pos_scaler.fit_transform(
            df_copy.loc[pos_mask, col].values.reshape(-1, 1)
        ).flatten()

    y = scaled_s
    
    
    #y = df_news[w]
    y_corr = df_copy[column_for_correlation]
    
    X_train, X_test, y_train, y_test, y_corr_train, y_corr_test = train_test_split(
        X, y, y_corr, test_size=0.2, random_state=13
    )
    
    model = SVR(kernel='poly', C=0.25, epsilon=0.001, degree=7, gamma='scale')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test))
    smape = 100/len(y_test) * np.sum(2 * np.abs(y_pred - y_test) / (np.abs(y_pred) + np.abs(y_test)))
    r2 = r2_score(y_test, y_pred)

    nonzero_mask = y_test != 0
    if np.any(nonzero_mask):
        mape = np.mean(np.abs((y_test[nonzero_mask] - y_pred[nonzero_mask]) / y_test[nonzero_mask])) * 100
    else:
        mape = np.nan

    print(f'{w} | y_pred var: {np.var(y_pred)} | y_corr_test var: {np.var(y_corr_test)} | smape: {smape}')

    
    corr_with_price_change_4h, _ = pearsonr(y_pred, y_corr_test)

    results.append({
        'Metric': w,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'SMAPE': smape,
        'R²': r2,
        f'Corr_with_{column_for_correlation}': corr_with_price_change_4h
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)
print(results_df)

# Plotting
plt.figure(figsize=(14, 6))
metrics_to_plot = ['RMSE', 'MAE']

# Bar plot for errors
results_df_melted = results_df.melt(id_vars='Metric', value_vars=metrics_to_plot)
sns.barplot(data=results_df_melted, x='Metric', y='value', hue='variable')
plt.xticks(rotation=45)
plt.title('Error Metrics by Model')
plt.ylabel('Error Value')
plt.grid(axis='y')
plt.show()

# Bar plot for R² and correlation
plt.figure(figsize=(14, 5))
sns.barplot(x='Metric', y='R²', data=results_df, color='skyblue', label='R²')
sns.barplot(x='Metric', y=f'Corr_with_{column_for_correlation}', data=results_df, color='salmon', label=f'Corr with {column_for_correlation}')
plt.xticks(rotation=45)
plt.title('R² and Correlation with price_change_4h')
plt.legend()
plt.grid(axis='y')
plt.show()



volume_surge | y_pred var: 0.0003374346188383076 | y_corr_test var: 0.00032534705581344173 | smape: 173.83421160662616


volume_surge_volatility | y_pred var: 0.0003004697775219677 | y_corr_test var: 0.0003143803850982776 | smape: 184.2852307271943


volume_surge_price | y_pred var: 4.114745716191647e-05 | y_corr_test var: 0.00032565548518349684 | smape: 154.14799083290998


price_change_4h_tanh | y_pred var: 0.00048373230509928433 | y_corr_test var: 0.0002914566549254941 | smape: 147.75103059946196


price_change_4h_volatility_tanh | y_pred var: 0.0004694719918324976 | y_corr_test var: 0.0003138442134536726 | smape: 147.84608410079264


price_change_4h_ema | y_pred var: 0.0004662724769710791 | y_corr_test var: 0.00029996742958460687 | smape: 147.63806019893573


price_change_4h_volatility_ema | y_pred var: 0.0006984274827653271 | y_corr_test var: 0.0003245637305680736 | smape: 147.08391797498152
                            Metric       MSE      RMSE       MAE  \
0                     volume_surge  0.014754  0.121468  0.074211   
1          volume_surge_volatility  0.014593  0.120804  0.070704   
2               volume_surge_price  0.001478  0.038446  0.012893   
3             price_change_4h_tanh  0.012584  0.112180  0.073817   
4  price_change_4h_volatility_tanh  0.013551  0.116411  0.076016   
5              price_change_4h_ema  0.012526  0.111921  0.073021   
6   price_change_4h_volatility_ema  0.016666  0.129097  0.083474   

           MAPE       SMAPE        R²  Corr_with_price_change_4h  
0  1.760653e+02  173.834212 -0.007582                   0.036096  
1  1.293736e+02  184.285231 -0.001594                   0.053308  
2  6.401209e+03  154.147991 -0.016614                   0.006324  
3  2.926250e+02  147.751031  0.006282              

/tmp/ipykernel_39456/528176635.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_39456/528176635.py:141: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
cleaned_titles = df_news['title'].apply(clean_text)
vectorizer = TfidfVectorizer(max_df=0.1, min_df=1, ngram_range=(1,7), sublinear_tf=False)
X = vectorizer.fit_transform(cleaned_titles)

scaler = MinMaxScaler(feature_range=(-1, 1))
s_scaled = scaler.fit_transform(df_news['volume_surge_price'].values.reshape(-1, 1))
y = pd.Series(s_scaled.flatten(), index=df_news[w].index)

#y = df_news['price_change_4h_volatility_ema']

X_train, X_test, y_train, y_test, _, X_demo = train_test_split(
    X, y, df_news['title'],  test_size=0.2, random_state=13
)

model = SVR(kernel='poly', C=0.25, epsilon=0.001, degree=7, gamma='scale')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [10]:
#_, X_demo, _, _ = train_test_split(df_news['title'], y, test_size=0.2, random_state=13)

df_demo = pd.DataFrame({"title": X_demo, "sentiment": y_test})

hello = df_demo.sort_values('sentiment', ascending=True).head(30)
for _, row in hello.iterrows():  
    print(f"{row['sentiment']} : {row['title']}")

-0.9397985320473834 : CME Group to close Chicago trading floor as a precaution due to coronavirus
-0.3373744794850798 : Whopping $520m liquidated so far on BitMEX and counting..
-0.07675822739856053 : New Bitcoin ETF proposal filed with SEC by Gold Fund Veteran
-0.07234917370919314 : Binance has added Shiba Inu/GBP trading pair.
-0.043212711540875914 : Fidelity-backed crypto security startup Fireblocks launches 'Secure Asset Transfer Network'
-0.00954753035569983 : European watchdogs demand detail on Facebook's cryptocurrency
0.017361611458127146 : There are now 11 blockchain unicorns in the world, with over $1B valuation
0.025875849996814998 : South Korea to launch 'Digital assets committee' in response to Terra implosion
0.026913049449967502 : Poloniex now allows fiat-to-crypto trading, credit card purchases
0.09656171014352977 : Major corporate NFT initiatives last year
0.12706261147237133 : $366 million liquidated from the cryptocurrency market in the past 60 minutes.
0.13347316506

## Label diagnostics

Computed on `dfn`, a time-sorted copy of `df_news`. All definitions above are used as coded.

1. Definitions — $\sigma_{\max}$, the volatility window and its rescaling, and the temporal
   granularity of each rolling and EMA statistic.
2. Distribution of each candidate label: variance, quantiles, proportion near zero.
3. Label comparison under time-aware cross-validation with out-of-sample $R^2$
   against training-mean and zero predictors, and correlation.
4. Concurrent headlines, event clusters, market-adjusted returns, and sensitivity to
   horizons of 1h/2h/4h/8h/24h and to excluding dense news intervals.
5. Look-ahead check — $\sigma_{\max}$ recomputed on an expanding historical window.

In [11]:
# ============================================================================
# Shared setup for the label diagnostics
# ============================================================================
import numpy as np
import pandas as pd

label_cols = {
    'dP_smooth':       'price_change_4h_tanh',             # \Delta P^{smooth}
    'dP_smoothVol':    'price_change_4h_volatility_tanh',  # \Delta P^{smoothVolatility}
    'dP_EMA':          'price_change_4h_ema',              # \Delta P^{EMA}
    'dP_EMAVol':       'price_change_4h_volatility_ema',   # \Delta P^{EMAVolatility}
    'V_surge':         'volume_surge',                     # V^{surge}
    'V_adjVolatility': 'volume_surge_volatility',          # V^{adjustedVolatility}
    'V_adjPrice':      'volume_surge_price',               # V^{adjustedPrice}
}

dfn = df_news.copy().sort_values('newsTimestamp').reset_index(drop=True)
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
print(f"analysis sample: {len(dfn)} headlines, "
      f"{pd.to_datetime(dfn['newsTimestamp'].min(), unit='s').date()} .. "
      f"{pd.to_datetime(dfn['newsTimestamp'].max(), unit='s').date()}")

analysis sample: 20271 headlines, 2019-01-03 .. 2025-03-28


In [12]:
# ============================================================================
# Definitions: constants read back from the code above.
# ============================================================================
BAR = 60                      # one row of `df` = 60 seconds
VOL_WINDOW = 30 * 60 * 60     # rolling window used for '30d_volatility'
ROLL_7D = 7 * 24 * 60         # rolling window for the 7-day volume averages
EMA_VOLUME = 4 * 60           # ewm span for the forward volume averages
EMA_LABEL = 60                # ewm span for the EMA labels
FWD_SHIFT, BACK_SHIFT = 15, 4 * 60   # shifts inside price_change_4h

sigma_max = df.loc[df['30d_volatility'].notna(), '30d_volatility'].max()

print("temporal granularity")
print(f"  base grid                       : 1 bar = {BAR}s (1-minute OHLCV)")
print(f"  volatility window               : {VOL_WINDOW:,} bars = {VOL_WINDOW/1440:.1f} days")
print(f"  volatility rescaling            : sd(log-return) x sqrt({VOL_WINDOW:,})")
print(f"                                    -> dimensionless log-return sd over a "
      f"{VOL_WINDOW/1440:.0f}-day horizon, NOT annualised")
print(f"  volume baseline (rolling mean)  : {ROLL_7D:,} bars = {ROLL_7D/1440:.0f} days")
print(f"  volume EMA (forward averages)   : span {EMA_VOLUME} bars = {EMA_VOLUME/60:.0f} hours")
print(f"  label EMA (dP_EMA / dP_EMAVol)  : span {EMA_LABEL} bars = {EMA_LABEL/60:.0f} hour,"
      f" computed on the 1-minute grid")
print(f"  price_change_4h                 : (close.shift(-{FWD_SHIFT}) - close.shift({BACK_SHIFT}))"
      f" / close  -> spans t-{BACK_SHIFT}min .. t+{FWD_SHIFT}min on the 1-minute grid")
print(f"  headline alignment              : last 1-minute bar at or before the headline "
      f"timestamp (searchsorted, side='right')")

print("\nsigma_max")
print(f"  value                           : {sigma_max:.6f}")
print(f"  estimator                       : maximum of '30d_volatility' over the full sample")
print(f"  attained at                     : {df['30d_volatility'].idxmax()}")
print(f"  volatility series range         : [{df['30d_volatility'].min():.6f}, "
      f"{sigma_max:.6f}], mean {df['30d_volatility'].mean():.6f}")
print(f"  enters the labels as            : (sigma_max - sigma_t), range "
      f"[0, {sigma_max - df['30d_volatility'].min():.6f}]")

temporal granularity
  base grid                       : 1 bar = 60s (1-minute OHLCV)
  volatility window               : 108,000 bars = 75.0 days
  volatility rescaling            : sd(log-return) x sqrt(108,000)
                                    -> dimensionless log-return sd over a 75-day horizon, NOT annualised
  volume baseline (rolling mean)  : 10,080 bars = 7 days
  volume EMA (forward averages)   : span 240 bars = 4 hours
  label EMA (dP_EMA / dP_EMAVol)  : span 60 bars = 1 hour, computed on the 1-minute grid
  price_change_4h                 : (close.shift(-15) - close.shift(240)) / close  -> spans t-240min .. t+15min on the 1-minute grid
  headline alignment              : last 1-minute bar at or before the headline timestamp (searchsorted, side='right')

sigma_max
  value                           : 0.990036
  estimator                       : maximum of '30d_volatility' over the full sample
  attained at                     : 2018-02-13 15:24:00
  volatility series range 

In [13]:
# ============================================================================
# Distribution of each candidate label: variance, quantiles, proportion near zero.
# ============================================================================
rows = []
for name, col in label_cols.items():
    x = dfn[col].dropna()
    rows.append({
        'label': name, 'n': len(x),
        'var': x.var(), 'std': x.std(),
        'p1': x.quantile(.01), 'q25': x.quantile(.25), 'median': x.median(),
        'q75': x.quantile(.75), 'p99': x.quantile(.99),
        'IQR': x.quantile(.75) - x.quantile(.25),
        'min': x.min(), 'max': x.max(),
        'frac<1e-2': (x.abs() < 1e-2).mean(),
        'frac<1e-3': (x.abs() < 1e-3).mean(),
        'skew': x.skew(), 'kurt': x.kurt(),
    })
dist = pd.DataFrame(rows).set_index('label')
dist['std_ratio'] = dist['std'] / dist['std'].max()
print(dist.round(5).to_string())
print(f"\nlargest / smallest standard deviation: "
      f"{dist['std'].max() / dist['std'].min():.1f}x  "
      f"({dist['std'].idxmax()} vs {dist['std'].idxmin()})")

                     n      var      std       p1      q25   median      q75      p99      IQR      min      max  frac<1e-2  frac<1e-3      skew       kurt  std_ratio
label                                                                                                                                                                 
dP_smooth        20271  0.00031  0.01769 -0.05198 -0.00627  0.00035  0.00695  0.05218  0.01322 -0.25498  0.16999    0.63505    0.09467  -1.00287   18.83237    0.20702
dP_smoothVol     20271  0.00014  0.01186 -0.03530 -0.00432  0.00025  0.00473  0.03386  0.00905 -0.17129  0.09497    0.75773    0.13463  -1.09360   18.66999    0.13882
dP_EMA           20271  0.00028  0.01670 -0.04845 -0.00595  0.00029  0.00640  0.04928  0.01235 -0.22870  0.12480    0.65591    0.10143  -1.12223   19.22030    0.19552
dP_EMAVol        20271  0.00013  0.01123 -0.03383 -0.00407  0.00020  0.00444  0.03263  0.00851 -0.15352  0.07777    0.77534    0.14671  -1.17977   19.23998    0.1314

In [14]:
# LaTeX version of the distribution table (Table: tab:label_distributions)
tex = dist[['n', 'std', 'p1', 'q25', 'median', 'q75', 'p99', 'frac<1e-2', 'skew', 'kurt']].copy()
tex.columns = ['$n$', 'SD', '$p_{1}$', '$q_{25}$', 'median', '$q_{75}$', '$p_{99}$',
               '$\\Pr(|y|<10^{-2})$', 'skew', 'kurt']
print(tex.to_latex(float_format='%.4f', escape=False))

\begin{tabular}{lrrrrrrrrrr}
\toprule
 & $n$ & SD & $p_{1}$ & $q_{25}$ & median & $q_{75}$ & $p_{99}$ & $\Pr(|y|<10^{-2})$ & skew & kurt \\
label &  &  &  &  &  &  &  &  &  &  \\
\midrule
dP_smooth & 20271 & 0.0177 & -0.0520 & -0.0063 & 0.0004 & 0.0069 & 0.0522 & 0.6350 & -1.0029 & 18.8324 \\
dP_smoothVol & 20271 & 0.0119 & -0.0353 & -0.0043 & 0.0002 & 0.0047 & 0.0339 & 0.7577 & -1.0936 & 18.6700 \\
dP_EMA & 20271 & 0.0167 & -0.0484 & -0.0060 & 0.0003 & 0.0064 & 0.0493 & 0.6559 & -1.1222 & 19.2203 \\
dP_EMAVol & 20271 & 0.0112 & -0.0338 & -0.0041 & 0.0002 & 0.0044 & 0.0326 & 0.7753 & -1.1798 & 19.2400 \\
V_surge & 20271 & 0.0854 & -0.2483 & -0.0264 & 0.0017 & 0.0330 & 0.2760 & 0.2150 & 0.1525 & 11.2287 \\
V_adjVolatility & 20271 & 0.0612 & -0.1782 & -0.0179 & 0.0012 & 0.0222 & 0.1995 & 0.2985 & 0.3338 & 13.6755 \\
V_adjPrice & 20271 & 0.0027 & -0.0051 & -0.0001 & 0.0000 & 0.0001 & 0.0060 & 0.9924 & -12.3514 & 529.0389 \\
\bottomrule
\end{tabular}



In [15]:
# ============================================================================
# Label comparison with scale-independent criteria.
#
# Screening model as in the label-selection cell above (TF-IDF over 1-7 grams +
# degree-7 polynomial SVR), evaluated with TimeSeriesSplit, the vectoriser fitted
# on the training fold only, and the labels taken on their own scale.
#
#   R2_vs_mean : out-of-sample R^2 against a training-mean predictor
#   R2_vs_zero : out-of-sample R^2 against a constant-zero predictor
#   corr_label : corr(prediction, label) on the held-out fold
#   corr_price : corr(prediction, realised price change) on the held-out fold
#
# Out-of-fold predictions are kept in `oof` and reused below.
# Runtime: roughly 10 minutes.
# ============================================================================
import re
import time
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVR
from sklearn.model_selection import TimeSeriesSplit

_stop_words = set(stopwords.words('english'))
_lemmatizer = WordNetLemmatizer()


def clean_text_cv(text):
    text = re.sub(r'[^\w\s]', '', re.sub(r'\d+', '', text.lower()))
    return ' '.join(_lemmatizer.lemmatize(w) for w in word_tokenize(text)
                    if w not in _stop_words)


dfn['clean_title'] = dfn['title'].apply(clean_text_cv)
X_text = dfn['clean_title'].values
price = dfn['price_change_4h'].values
splits = list(TimeSeriesSplit(n_splits=5).split(X_text))

def run_scale_free_cv(labels, frame):
    """Time-aware CV of the screening model for each {name: column} in `labels`.

    Returns (metrics table, out-of-fold prediction frame).
    """
    oof_ = pd.DataFrame(index=frame.index)
    rows_ = []
    t0 = time.time()
    for name, col in labels.items():
        y = frame[col].values
        pred_oof = np.full(len(frame), np.nan)
        fold = {'corr_label': [], 'corr_price': [], 'R2_vs_mean': [], 'R2_vs_zero': [],
                'RMSE': [], 'MAE': []}
        for tr, te in splits:
            vec = TfidfVectorizer(max_df=0.1, min_df=1, ngram_range=(1, 7), sublinear_tf=False)
            Xtr = vec.fit_transform(X_text[tr])
            Xte = vec.transform(X_text[te])
            model = SVR(kernel='poly', degree=7, C=0.25, epsilon=0.001, gamma='scale',
                        cache_size=1000)
            model.fit(Xtr, y[tr])
            pred = model.predict(Xte)
            pred_oof[te] = pred
            yte = y[te]
            ss_res = np.sum((yte - pred) ** 2)
            fold['R2_vs_mean'].append(1 - ss_res / np.sum((yte - y[tr].mean()) ** 2))
            fold['R2_vs_zero'].append(1 - ss_res / np.sum(yte ** 2))
            fold['RMSE'].append(np.sqrt(ss_res / len(yte)))
            fold['MAE'].append(np.mean(np.abs(yte - pred)))
            if pred.std() > 0:
                fold['corr_label'].append(np.corrcoef(pred, yte)[0, 1])
                fold['corr_price'].append(np.corrcoef(pred, price[te])[0, 1])
        oof_[name] = pred_oof
        m = {k: (np.mean(v) if len(v) else np.nan) for k, v in fold.items()}
        m['label'] = name
        m['n'] = int(np.isfinite(pred_oof).sum())
        rows_.append(m)
        print(f"{name:20s} corr_label={m['corr_label']:+.4f}  corr_price={m['corr_price']:+.4f}"
              f"  R2_vs_mean={m['R2_vs_mean']:+.4f}  R2_vs_zero={m['R2_vs_zero']:+.4f}"
              f"  RMSE={m['RMSE']:.4f}  MAE={m['MAE']:.4f}   [{time.time()-t0:.0f}s]",
              flush=True)
    table = pd.DataFrame(rows_).set_index('label')[
        ['n', 'corr_label', 'corr_price', 'R2_vs_mean', 'R2_vs_zero', 'RMSE', 'MAE']]
    return table, oof_


scale_free, oof = run_scale_free_cv(label_cols, dfn)
print()
print(scale_free.round(5).to_string())
print("\nrank by R2_vs_mean :", list(scale_free['R2_vs_mean'].sort_values(ascending=False).index))
print("rank by corr_price :", list(scale_free['corr_price'].sort_values(ascending=False).index))
print("rank by RMSE (low first):",
      list(scale_free['RMSE'].sort_values().index))

dP_smooth            corr_label=+0.0458  corr_price=+0.0458  R2_vs_mean=-0.0092  R2_vs_zero=-0.0106  RMSE=0.0170  MAE=0.0113   [88s]


dP_smoothVol         corr_label=+0.0465  corr_price=+0.0473  R2_vs_mean=-0.0060  R2_vs_zero=-0.0076  RMSE=0.0116  MAE=0.0077   [171s]


dP_EMA               corr_label=+0.0662  corr_price=+0.0489  R2_vs_mean=-0.0069  R2_vs_zero=-0.0085  RMSE=0.0159  MAE=0.0106   [263s]


dP_EMAVol            corr_label=+0.0669  corr_price=+0.0498  R2_vs_mean=-0.0031  R2_vs_zero=-0.0048  RMSE=0.0109  MAE=0.0072   [353s]


V_surge              corr_label=+0.0202  corr_price=+0.0394  R2_vs_mean=-0.0150  R2_vs_zero=-0.0136  RMSE=0.0795  MAE=0.0496   [471s]


V_adjVolatility      corr_label=+0.0215  corr_price=+0.0414  R2_vs_mean=-0.0141  R2_vs_zero=-0.0133  RMSE=0.0579  MAE=0.0352   [584s]


V_adjPrice           corr_label=+0.0320  corr_price=+0.0556  R2_vs_mean=-0.2184  R2_vs_zero=-0.2288  RMSE=0.0020  MAE=0.0011   [599s]



                     n  corr_label  corr_price  R2_vs_mean  R2_vs_zero     RMSE      MAE
label                                                                                   
dP_smooth        16890     0.04584     0.04584    -0.00919    -0.01059  0.01695  0.01130
dP_smoothVol     16890     0.04652     0.04733    -0.00604    -0.00756  0.01157  0.00771
dP_EMA           16890     0.06621     0.04886    -0.00693    -0.00854  0.01593  0.01060
dP_EMAVol        16890     0.06687     0.04984    -0.00306    -0.00480  0.01088  0.00723
V_surge          16890     0.02020     0.03942    -0.01497    -0.01356  0.07954  0.04960
V_adjVolatility  16890     0.02154     0.04143    -0.01412    -0.01334  0.05788  0.03520
V_adjPrice       16890     0.03197     0.05559    -0.21840    -0.22876  0.00204  0.00109

rank by R2_vs_mean : ['dP_EMAVol', 'dP_smoothVol', 'dP_EMA', 'dP_smooth', 'V_adjVolatility', 'V_surge', 'V_adjPrice']
rank by corr_price : ['V_adjPrice', 'dP_EMAVol', 'dP_EMA', 'dP_smoothVol', 'dP_

In [16]:
tex2 = scale_free[['corr_price', 'R2_vs_mean', 'R2_vs_zero', 'RMSE', 'MAE']].copy()
tex2.columns = ['corr. with $\\Delta P_{4h}$', '$R^2$ vs.\\ mean', '$R^2$ vs.\\ zero',
                'RMSE', 'MAE']
print(tex2.to_latex(float_format='%.4f', escape=False))

\begin{tabular}{lrrrrr}
\toprule
 & corr. with $\Delta P_{4h}$ & $R^2$ vs.\ mean & $R^2$ vs.\ zero & RMSE & MAE \\
label &  &  &  &  &  \\
\midrule
dP_smooth & 0.0458 & -0.0092 & -0.0106 & 0.0170 & 0.0113 \\
dP_smoothVol & 0.0473 & -0.0060 & -0.0076 & 0.0116 & 0.0077 \\
dP_EMA & 0.0489 & -0.0069 & -0.0085 & 0.0159 & 0.0106 \\
dP_EMAVol & 0.0498 & -0.0031 & -0.0048 & 0.0109 & 0.0072 \\
V_surge & 0.0394 & -0.0150 & -0.0136 & 0.0795 & 0.0496 \\
V_adjVolatility & 0.0414 & -0.0141 & -0.0133 & 0.0579 & 0.0352 \\
V_adjPrice & 0.0556 & -0.2184 & -0.2288 & 0.0020 & 0.0011 \\
\bottomrule
\end{tabular}



In [17]:
# ============================================================================
# Headline crowding and event clusters
# ============================================================================
ts = dfn['newsTimestamp'].astype('int64').values
win = 4 * 3600

ends = np.searchsorted(ts, ts + win, side='right')
dfn['n_concurrent'] = ends - np.arange(len(ts)) - 1
print("other headlines sharing the same forward 4h window:")
print(dfn['n_concurrent'].describe().round(3).to_string())
print("quantiles:", {k: round(v, 1) for k, v in
                     dfn['n_concurrent'].quantile([.5, .75, .9, .95, .99]).to_dict().items()})
print("share of headlines with at least one other headline in the window:",
      round((dfn['n_concurrent'] > 0).mean(), 4))

gap = np.diff(ts, prepend=ts[0])
dfn['cluster_id'] = (gap > win).cumsum()
n_clusters = dfn['cluster_id'].nunique()
sizes = dfn.groupby('cluster_id').size()
print(f"\nheadlines: {len(dfn)}   event clusters: {n_clusters}   "
      f"mean headlines per cluster: {len(dfn)/n_clusters:.2f}")
print("cluster size distribution:")
print(sizes.describe().round(3).to_string())
print("isolated headlines (cluster size 1):", int((sizes == 1).sum()))

other headlines sharing the same forward 4h window:
count    20271.000
mean         3.215
std          2.963
min          0.000
25%          1.000
50%          3.000
75%          5.000
max         38.000
quantiles: {0.5: 3.0, 0.75: 5.0, 0.9: 7.0, 0.95: 8.0, 0.99: 13.0}
share of headlines with at least one other headline in the window: 0.8563

headlines: 20271   event clusters: 2913   mean headlines per cluster: 6.96
cluster size distribution:
count    2913.000
mean        6.959
std        10.045
min         1.000
25%         2.000
50%         4.000
75%         9.000
max       226.000
isolated headlines (cluster size 1): 688


In [18]:
# ============================================================================
# Market-adjusted (abnormal) return.
#
# Macro controls: hourly S&P 500, gold and VXX bars (backtesting/data/cleaned).
# For each headline the macro return is measured over the SAME window as the
# Bitcoin return the label is built from, using the last macro bar at or before
# each window endpoint. Headlines whose window falls entirely outside equity
# trading hours have no usable macro move and are dropped, which is why the
# regression sample is much smaller than the full headline sample.
#
# Caveat: VXX is a VIX-futures ETN, used here as the volatility proxy.
# ============================================================================
import statsmodels.api as sm

MACRO_DIR = "../backtesting/data/cleaned"
macro = {}
for key, fn in [('sp500', 'SP500_hourly.csv'), ('gold', 'GOLD_hourly.csv'),
                ('vix', 'VXX_hourly.csv')]:
    m = pd.read_csv(f"{MACRO_DIR}/{fn}").sort_values('timestamp').reset_index(drop=True)
    macro[key] = m
    print(f"{key:6s} {len(m):5d} hourly bars  "
          f"{pd.to_datetime(m['timestamp'].min(), unit='s').date()} .. "
          f"{pd.to_datetime(m['timestamp'].max(), unit='s').date()}")

MAX_STALENESS = 6 * 3600  # a macro bar older than this is treated as unusable


def macro_return(key, t0, t1):
    """Return of macro series `key` between the last bars at or before t0 and t1."""
    m = macro[key]
    mt, mc = m['timestamp'].values, m['close'].values
    i0 = np.searchsorted(mt, t0, side='right') - 1
    i1 = np.searchsorted(mt, t1, side='right') - 1
    ok = ((i0 >= 0) & (i1 >= 0) & (i1 > i0)
          & (t0 - mt[np.clip(i0, 0, len(mt) - 1)] <= MAX_STALENESS)
          & (t1 - mt[np.clip(i1, 0, len(mt) - 1)] <= MAX_STALENESS))
    c0 = mc[np.clip(i0, 0, len(mc) - 1)]
    c1 = mc[np.clip(i1, 0, len(mc) - 1)]
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(ok, (c1 - c0) / c0, np.nan)


# `price_change_4h` as published spans t-4h .. t+15min, so the macro controls
# are measured over that same span.
t_news = dfn['newsTimestamp'].values.astype('int64')
for key in ['sp500', 'gold', 'vix']:
    dfn[f'{key}_ret'] = macro_return(key, t_news - 4 * 3600, t_news + 15 * 60)
dfn = dfn.rename(columns={'vix_ret': 'vix_chg'})

macro_cols = ['sp500_ret', 'gold_ret', 'vix_chg']
d = dfn.dropna(subset=['price_change_4h'] + macro_cols).copy()
print(f"\nheadlines with a usable contemporaneous macro move: {len(d)} / {len(dfn)} "
      f"({len(d)/len(dfn):.1%})")

fit = sm.OLS(d['price_change_4h'], sm.add_constant(d[macro_cols])).fit()
print(fit.summary().tables[1])
print(f"macro model R^2 on the 4h BTC return: {fit.rsquared:.4f}")
d['abn_ret'] = fit.resid

print("\n[i] candidate label vs return:")
abn = pd.DataFrame({
    'corr_raw': {n: d[c].corr(d['price_change_4h']) for n, c in label_cols.items()},
    'corr_abnormal': {n: d[c].corr(d['abn_ret']) for n, c in label_cols.items()},
})
abn['delta'] = abn['corr_abnormal'] - abn['corr_raw']
print(abn.round(5).to_string())

print("\n[ii] out-of-fold model prediction vs return:")
dm = d
abn_p = pd.DataFrame({
    'corr_raw': {n: oof.loc[dm.index, n].corr(dm['price_change_4h']) for n in label_cols},
    'corr_abnormal': {n: oof.loc[dm.index, n].corr(dm['abn_ret']) for n in label_cols},
    'n': {n: int(oof.loc[dm.index, n].notna().sum()) for n in label_cols},
})
abn_p['delta'] = abn_p['corr_abnormal'] - abn_p['corr_raw']
print(abn_p.round(5).to_string())

sp500   9390 hourly bars  2019-09-30 .. 2025-09-26
gold    9390 hourly bars  2019-09-30 .. 2025-09-26
vix     9390 hourly bars  2019-09-30 .. 2025-09-26



headlines with a usable contemporaneous macro move: 4526 / 20271 (22.3%)


                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0008      0.000     -3.235      0.001      -0.001      -0.000
sp500_ret      1.1411      0.053     21.670      0.000       1.038       1.244
gold_ret       0.2836      0.073      3.893      0.000       0.141       0.426
vix_chg       -0.0132      0.013     -1.017      0.309      -0.039       0.012
macro model R^2 on the 4h BTC return: 0.1981

[i] candidate label vs return:


                 corr_raw  corr_abnormal    delta
dP_smooth         1.00000        0.89546 -0.10454
dP_smoothVol      0.99177        0.88737 -0.10440
dP_EMA            0.94253        0.84631 -0.09622
dP_EMAVol         0.93493        0.83894 -0.09598
V_surge           0.50240        0.43940 -0.06300
V_adjVolatility   0.48482        0.42454 -0.06028
V_adjPrice        0.58545        0.53819 -0.04726

[ii] out-of-fold model prediction vs return:
                 corr_raw  corr_abnormal     n    delta
dP_smooth         0.03891        0.03522  4343 -0.00368
dP_smoothVol      0.04076        0.03857  4343 -0.00220
dP_EMA            0.04179        0.03449  4343 -0.00730
dP_EMAVol         0.04271        0.03693  4343 -0.00578
V_surge           0.03000        0.03505  4343  0.00506
V_adjVolatility   0.02768        0.03290  4343  0.00523
V_adjPrice        0.04099        0.04807  4343  0.00709


In [19]:
# ============================================================================
# One observation per event cluster.
# ============================================================================
clu = pd.DataFrame({
    'label_headline': {n: dfn[c].corr(dfn['price_change_4h']) for n, c in label_cols.items()},
    'label_cluster': {n: dfn.groupby('cluster_id')[[c, 'price_change_4h']].mean()
                      .corr().iloc[0, 1] for n, c in label_cols.items()},
})
pred_frame = dfn[['cluster_id', 'price_change_4h']].join(oof)
clu['pred_headline'] = pd.Series({n: pred_frame[n].corr(pred_frame['price_change_4h']) for n in label_cols})
clu['pred_cluster'] = pd.Series({n: pred_frame.groupby('cluster_id')[[n, 'price_change_4h']].mean()
                                 .corr().iloc[0, 1] for n in label_cols})
print("corr with price_change_4h, per headline vs per event cluster")
print("(label_* = candidate label; pred_* = out-of-fold model prediction)")
print(clu.round(5).to_string())
print("\nclusters with at least one out-of-fold prediction:",
      int(pred_frame.dropna(subset=['V_adjPrice'])['cluster_id'].nunique()))

solo = pred_frame[pred_frame['cluster_id'].isin(sizes[sizes == 1].index)]
print(f"\nisolated headlines only (n={len(solo)}), out-of-fold prediction vs return:")
print(pd.Series({n: solo[n].corr(solo['price_change_4h']) for n in label_cols}).round(5).to_string())

corr with price_change_4h, per headline vs per event cluster
(label_* = candidate label; pred_* = out-of-fold model prediction)
                 label_headline  label_cluster  pred_headline  pred_cluster
dP_smooth               0.99999        0.99999        0.04071       0.11131
dP_smoothVol            0.98831        0.98731        0.04150       0.10293
dP_EMA                  0.94258        0.97273        0.04407       0.11656
dP_EMAVol               0.93241        0.96156        0.04474       0.10957
V_surge                 0.49675        0.52238        0.02944       0.09305
V_adjVolatility         0.47920        0.50459        0.02779       0.08360
V_adjPrice              0.57043        0.64192        0.02620       0.02633

clusters with at least one out-of-fold prediction: 2318

isolated headlines only (n=688), out-of-fold prediction vs return:
dP_smooth          0.07792
dP_smoothVol       0.07151
dP_EMA             0.08440
dP_EMAVol          0.08299
V_surge            0.10426
V_ad

In [20]:
# ============================================================================
# Sensitivity to the response horizon and to concurrent headlines.
# Forward returns are measured on the 1-minute frame as
#     ret_h = (close[t+h] - close[t]) / close[t]
# and mapped back onto the headlines.
# ============================================================================
HORIZONS = [60, 120, 240, 480, 1440]
idx = np.clip(np.searchsorted(df['timestamp'].values, dfn['newsTimestamp'].values,
                              side='right') - 1, 0, len(df) - 1)
for h in HORIZONS:
    df[f'ret_{h}'] = (df['close'].shift(-h) - df['close']) / df['close']
    dfn[f'ret_{h}'] = df.iloc[idx][f'ret_{h}'].values

pred_frame = dfn[['cluster_id', 'n_concurrent', 'price_change_4h']
                 + [f'ret_{h}' for h in HORIZONS]].join(oof)

hz = pd.DataFrame({f'{h//60}h': {n: pred_frame[n].corr(pred_frame[f'ret_{h}']) for n in label_cols}
                   for h in HORIZONS})
hz['as_published'] = pd.Series({n: pred_frame[n].corr(pred_frame['price_change_4h']) for n in label_cols})
print("out-of-fold prediction vs realised forward return, by horizon:")
print(hz.round(5).to_string())
print("\nbest label per horizon (largest |corr|):", hz.abs().idxmax().to_dict())

# the same sensitivity at the level of the label itself, not the model
hz_lab = pd.DataFrame({f'{h//60}h': {n: dfn[c].corr(dfn[f'ret_{h}'])
                                     for n, c in label_cols.items()} for h in HORIZONS})
hz_lab['as_published'] = pd.Series({n: dfn[c].corr(dfn['price_change_4h'])
                                    for n, c in label_cols.items()})
print("\nlabel vs realised forward return, by horizon:")
print(hz_lab.round(5).to_string())

thr = dfn['n_concurrent'].quantile(0.90)
sparse = pred_frame[pred_frame['n_concurrent'] <= thr]
print(f"\nexcluding the densest 10% of news windows (n_concurrent > {thr:.0f}): "
      f"{len(sparse)} of {len(dfn)} headlines retained")
sp = pd.DataFrame({
    'all_asPublished': {n: pred_frame[n].corr(pred_frame['price_change_4h']) for n in label_cols},
    'sparse_asPublished': {n: sparse[n].corr(sparse['price_change_4h']) for n in label_cols},
    'all_ret240': {n: pred_frame[n].corr(pred_frame['ret_240']) for n in label_cols},
    'sparse_ret240': {n: sparse[n].corr(sparse['ret_240']) for n in label_cols},
})
print(sp.round(5).to_string())

out-of-fold prediction vs realised forward return, by horizon:
                      1h       2h       4h       8h      24h  as_published
dP_smooth       -0.00413 -0.00047 -0.00265 -0.00193  0.00196       0.04071
dP_smoothVol    -0.00484 -0.00102 -0.00280 -0.00354  0.00118       0.04150
dP_EMA          -0.00526 -0.00106 -0.00483 -0.00062 -0.00149       0.04407
dP_EMAVol       -0.00646 -0.00224 -0.00434 -0.00053  0.00036       0.04474
V_surge          0.00304  0.00369  0.00063  0.00520  0.01119       0.02944
V_adjVolatility  0.00213  0.00317  0.00100  0.00482  0.01324       0.02779
V_adjPrice      -0.00682 -0.01432 -0.01133 -0.00271  0.00419       0.02620

best label per horizon (largest |corr|): {'1h': 'V_adjPrice', '2h': 'V_adjPrice', '4h': 'V_adjPrice', '8h': 'V_surge', '24h': 'V_adjVolatility', 'as_published': 'dP_EMAVol'}

label vs realised forward return, by horizon:
                      1h       2h       4h       8h      24h  as_published
dP_smooth        0.10150  0.06417  0.025

In [21]:
# ============================================================================
# sigma_max estimated on an expanding historical window.
#
# `max_vol` above is the maximum of '30d_volatility' over the full sample. This cell
# leaves that definition in place and adds a counterpart in which sigma_max is the
# expanding maximum
#     sigma_max(t) = max_{s <= t} sigma_s   (pandas cummax),
# which uses only information available at t. Only the four volatility-rescaled
# labels are affected; the remaining three are unchanged by construction.
# ============================================================================
vol = df['30d_volatility']
sigma_max_global = vol.max()
argmax_time = vol.idxmax()
argmax_ts = int(df.loc[df.index == argmax_time, 'timestamp'].iloc[0])

print(f"sigma_max (full sample)      : {sigma_max_global:.6f}, attained {argmax_time}")
share_before = (dfn['newsTimestamp'] < argmax_ts).mean()
print(f"headlines dated before that  : {share_before:.1%} of the sample")

df['sigma_max_expanding'] = vol.cummax()
dfn['sigma_max_expanding'] = df.iloc[idx]['sigma_max_expanding'].values
print(f"expanding sigma_max at headline times: min {dfn['sigma_max_expanding'].min():.6f}, "
      f"median {dfn['sigma_max_expanding'].median():.6f}, max {dfn['sigma_max_expanding'].max():.6f}")
print(f"headlines where expanding == global  : "
      f"{np.isclose(dfn['sigma_max_expanding'], sigma_max_global).mean():.1%}")

# --- expanding-window counterparts of the volatility-rescaled labels ---------
mexp = df['sigma_max_expanding']
df['price_change_4h_volatility_tanh_c'] = np.tanh(df['price_change_4h'] * (mexp - vol))
df['price_change_4h_volatility_ema_c'] = (
    df['price_change_4h_volatility_tanh_c'].ewm(span=EMA_LABEL, adjust=False).mean())
_adj_c = df['raw_volume_surge'] * (mexp - vol + 1e-8)
df['volume_surge_volatility_c'] = np.tanh(_adj_c)
df['volume_surge_price_c'] = np.tanh(_adj_c * df['price_change_4h'].abs())

causal_cols = {
    'dP_smoothVol_c':    'price_change_4h_volatility_tanh_c',
    'dP_EMAVol_c':       'price_change_4h_volatility_ema_c',
    'V_adjVolatility_c': 'volume_surge_volatility_c',
    'V_adjPrice_c':      'volume_surge_price_c',
}
for name, col in causal_cols.items():
    dfn[col] = df.iloc[idx][col].values

pairs = [('dP_smoothVol', 'dP_smoothVol_c'), ('dP_EMAVol', 'dP_EMAVol_c'),
         ('V_adjVolatility', 'V_adjVolatility_c'), ('V_adjPrice', 'V_adjPrice_c')]
cmp_rows = []
for pub, cau in pairs:
    a, b = dfn[label_cols[pub]], dfn[causal_cols[cau]]
    cmp_rows.append({'label': pub, 'corr(published, causal)': a.corr(b),
                     'std_published': a.std(), 'std_causal': b.std(),
                     'mean_abs_diff': (a - b).abs().mean(),
                     'max_abs_diff': (a - b).abs().max()})
print("\npublished (full-sample sigma_max) vs causal (expanding sigma_max):")
print(pd.DataFrame(cmp_rows).set_index('label').round(6).to_string())

# --- scale-free comparison on the causally rescaled labels -------------------
print("\nscale-free comparison on the expanding-window labels:")
scale_free_c, oof_c = run_scale_free_cv(causal_cols, dfn)
print()
side = pd.concat([
    scale_free.loc[[p for p, _ in pairs], ['corr_price', 'R2_vs_mean', 'RMSE']]
        .add_suffix('_published'),
    scale_free_c[['corr_price', 'R2_vs_mean', 'RMSE']].add_suffix('_causal')
        .set_axis([p for p, _ in pairs]),
], axis=1)
print(side.round(5).to_string())
print("\npublished ranking by corr_price:", list(scale_free['corr_price'].sort_values(ascending=False).index))
print("causal   ranking by corr_price:", list(scale_free_c['corr_price'].sort_values(ascending=False).index))
print("published ranking by R2_vs_mean:", list(scale_free['R2_vs_mean'].sort_values(ascending=False).index))
print("causal   ranking by R2_vs_mean:", list(scale_free_c['R2_vs_mean'].sort_values(ascending=False).index))

sigma_max (full sample)      : 0.990036, attained 2018-02-13 15:24:00
headlines dated before that  : 0.0% of the sample
expanding sigma_max at headline times: min 0.990036, median 0.990036, max 0.990036
headlines where expanding == global  : 100.0%



published (full-sample sigma_max) vs causal (expanding sigma_max):
                 corr(published, causal)  std_published  std_causal  mean_abs_diff  max_abs_diff
label                                                                                           
dP_smoothVol                         1.0       0.011860    0.011860            0.0           0.0
dP_EMAVol                            1.0       0.011228    0.011228            0.0           0.0
V_adjVolatility                      1.0       0.061247    0.061247            0.0           0.0
V_adjPrice                           1.0       0.002747    0.002747            0.0           0.0

scale-free comparison on the expanding-window labels:


dP_smoothVol_c       corr_label=+0.0465  corr_price=+0.0473  R2_vs_mean=-0.0060  R2_vs_zero=-0.0076  RMSE=0.0116  MAE=0.0077   [92s]


dP_EMAVol_c          corr_label=+0.0669  corr_price=+0.0498  R2_vs_mean=-0.0031  R2_vs_zero=-0.0048  RMSE=0.0109  MAE=0.0072   [183s]


V_adjVolatility_c    corr_label=+0.0215  corr_price=+0.0414  R2_vs_mean=-0.0141  R2_vs_zero=-0.0133  RMSE=0.0579  MAE=0.0352   [297s]


V_adjPrice_c         corr_label=+0.0320  corr_price=+0.0556  R2_vs_mean=-0.2184  R2_vs_zero=-0.2288  RMSE=0.0020  MAE=0.0011   [313s]



                 corr_price_published  R2_vs_mean_published  RMSE_published  corr_price_causal  R2_vs_mean_causal  RMSE_causal
dP_smoothVol                  0.04733              -0.00604         0.01157            0.04733           -0.00604      0.01157
dP_EMAVol                     0.04984              -0.00306         0.01088            0.04984           -0.00306      0.01088
V_adjVolatility               0.04143              -0.01412         0.05788            0.04143           -0.01412      0.05788
V_adjPrice                    0.05559              -0.21840         0.00204            0.05559           -0.21840      0.00204

published ranking by corr_price: ['V_adjPrice', 'dP_EMAVol', 'dP_EMA', 'dP_smoothVol', 'dP_smooth', 'V_adjVolatility', 'V_surge']
causal   ranking by corr_price: ['V_adjPrice_c', 'dP_EMAVol_c', 'dP_smoothVol_c', 'V_adjVolatility_c']
published ranking by R2_vs_mean: ['dP_EMAVol', 'dP_smoothVol', 'dP_EMA', 'dP_smooth', 'V_adjVolatility', 'V_surge', 'V_adjPrice']
